# CNN-EXP-035 — PER-BASE BIG VERIFIERNET

## Решение, которое фиксируем

С этого эксперимента **перестаём обучать модель искать 32-bp окно**.

Единица прогноза:

```text
1 nucleotide × 1 strand
```

То есть модель должна ответить непосредственно:

```text
DNA position i
      ↓
initiation score / rate для этой конкретной bp
```

Никакого `region positive`, `max inside 32 bp` или regional BCE в objective нет.

---

## Главная гипотеза EXP035

Нам уже недостаточно одной модели, которая спрашивает:

> «Похоже ли это место на TSS?»

Добавляем второй механизм:

> **Context Verifier:** «Если backbone считает эту bp TSS, подтверждает ли широкий
> контекст это решение, или это promoter-like FP? И наоборот — не пропустил ли
> backbone настоящий TSS?»

Verifier получает:

- локальные promoter features;
- несколько масштабов широкого genomic context;
- текущий `base_log_rate`, который он должен исправлять.

Он выдаёт **signed correction**:

```text
base_log_rate
      +
context correction
      =
final_log_rate
```

Verifier обучается с повышенным весом на позициях, где базовая модель уже сейчас
сильно ошибается по rate. Это online hard-example emphasis.

---

## Архитектура

| Блок | Параметры | Разрешение | Роль |
|---|---|---:|---|
| Input | 5 каналов текущего pipeline | 8192 bp | raw DNA one-hot + N |
| Stem | Conv1d `k=11`, `5→96` | 1 bp | первичное sequence representation |
| Local block 1 | residual `k=5, d=1` | 1 bp | короткие motifs |
| Local block 2 | residual `k=5, d=2` | 1 bp | promoter grammar, local RF ≈35 bp |
| Tower L1 | stride-4 Conv9, `96→128`, residual `d=1,3` | 4 bp | ближний context |
| Tower L2 | stride-4 Conv9, `128→192`, residual `d=1,3` | 16 bp | medium context |
| Tower L3 | stride-4 Conv9, `192→256`, residual `d=1,3` | 64 bp | regional context |
| Tower L4 | stride-4 Conv9, `256→320`, residual `d=1,3` | 256 bp | long context |
| Tower L5 | stride-4 Conv9, `320→384`, residual `d=1,3` | 1024 bp | very long context |
| Multi-scale fusion | `1×1 → 96`, upsample всех levels | 1 bp | объединяет масштабы |
| Base rate head | `96→96→1` | 1 bp | основной log-rate |
| Context Verifier | local + L3 + L4 + L5 + detached base score | 1 bp | ищет FP/FN по косвенным context cues |
| Correction | `2 × tanh(verifier_logit / 3)` | 1 bp | ограниченная signed поправка |
| Final output | `base_log_rate + correction` | 1 bp | итоговый EPIC score source |

### Training window

```text
8192 bp input
│ 512 bp ignored │──── 7168 bp loss ────│ 512 bp ignored │
```

Каждая обучаемая nucleotide получает минимум 512 bp реального контекста по обе стороны.

---

## Loss

Три составляющие:

```text
1. Poisson(final_log_rate, count)     — основной objective
2. Poisson(base_log_rate, count)      — не даёт verifier заменить backbone
3. Hard-weighted verifier BCE         — специально учит исправлять FP/FN
```

Итог:

```text
L =
    L_final_poisson
  + 0.25 * L_base_poisson
  + 0.05 * L_verifier
```

Это уже **candidate architecture**, а не чистая one-change ablation.
Если она даст прирост, потом отдельно разберём ablation'ами, какой блок его дал.


In [1]:
from pathlib import Path
import copy
import gc
import json
import math
import sys
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_directml

from scipy.stats import pearsonr, rankdata

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    candidate
    for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    DirectMLAdam,
    ProfileWindowConfig,
    ProfileWindowGenerator,
)
from ml_project.epic_data import load_split

print("Project:", PROJECT_ROOT)


Project: D:\Projects\EPIC\EPIC


In [2]:
device = torch_directml.device()

print("DirectML device:", device)
print("GPU:", torch_directml.device_name(0))


DirectML device: privateuseone:0
GPU: AMD Radeon RX 6750 GRE 12GB 


In [3]:
# ============================================================
# DATA — PER-BASE ONLY
# ============================================================

split = load_split(PROJECT_ROOT)
sequences, fasta_alphabet = load_baseline_sequences(split)

train_contigs = tuple(
    sorted(set(split.intervals("train")["contig"].astype(str)))
)

validation_contigs = tuple(
    sorted(set(split.intervals("validation")["contig"].astype(str)))
)

assert set(train_contigs).isdisjoint(validation_contigs)

TARGET_LENGTH = 7168
CONTEXT_FLANK = 512
INPUT_LENGTH = TARGET_LENGTH + 2 * CONTEXT_FLANK

assert INPUT_LENGTH == 8192
assert INPUT_LENGTH % (4 ** 5) == 0

profile_config = ProfileWindowConfig(
    target_length=TARGET_LENGTH,
    context_flank=CONTEXT_FLANK,
    positive_branch_fraction=0.5,
)

train_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)

validation_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=profile_config,
    seed=42,
)

validation_tiles = validation_generator.tile_index.copy()

display(split.summary)

print("Train contigs:", len(train_contigs))
print("Validation contigs:", len(validation_contigs))
print("Input:", INPUT_LENGTH)
print("Per-base target:", TARGET_LENGTH)
print("Flank:", CONTEXT_FLANK)

smoke_batch = train_generator.sample_batch(batch_size=2)

print()
print("x_both_strands:", tuple(smoke_batch.x_both_strands.shape))
print("count:", tuple(smoke_batch.count.shape))
print("mask:", tuple(smoke_batch.mask.shape))

assert smoke_batch.x_both_strands.shape == (
    4,
    5,
    INPUT_LENGTH,
)

assert smoke_batch.count.shape[-1] == TARGET_LENGTH


,split,contigs,intervals_both_strands,whitelist_bp,position_strand_count,percent_of_all_positions,positive_positions,negative_positions,positive_percent
0,train,9,168998,114032080,228064160,59.316784,170443.0,227893717.0,0.074735
1,validation,3,60386,41903743,83807486,21.797334,56205.0,83751281.0,0.067064
2,test,3,54846,36306695,72613390,18.885882,NaN,NaN,NaN


Train contigs: 9
Validation contigs: 3
Input: 8192
Per-base target: 7168
Flank: 512

x_both_strands: (4, 5, 8192)
count: (2, 2, 7168)
mask: (2, 2, 7168)


## Модель

Важное отличие от EXP034:

```text
EXP034:
local RF35 + full-resolution dilated RF2053

EXP035:
local RF35
+
настоящий 5-level multi-scale tower
+
отдельный Context Verifier
```

Пятый уровень теоретически видит область шире самого 8-kb окна, поэтому фактический
контекст ограничен входным окном. Это нормально: verifier получает summary практически
всего окна, но итоговый prediction остаётся base-resolution.


In [4]:
# ============================================================
# MODEL
# ============================================================

CH_LOCAL = 96
TOWER_CHANNELS = (128, 192, 256, 320, 384)
CORRECTION_SCALE = 2.0


class SameResolutionResidual(nn.Module):
    def __init__(
        self,
        channels,
        *,
        kernel_size=5,
        dilation=1,
    ):
        super().__init__()

        padding = (
            dilation
            * (kernel_size - 1)
            // 2
        )

        self.bn1 = nn.BatchNorm1d(channels)
        self.conv1 = nn.Conv1d(
            channels,
            channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=padding,
        )

        self.bn2 = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(
            channels,
            channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=padding,
        )

    def forward(self, x):
        residual = x

        x = self.conv1(
            F.gelu(
                self.bn1(x)
            )
        )

        x = self.conv2(
            F.gelu(
                self.bn2(x)
            )
        )

        return x + residual


class TowerLevel(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        self.down = nn.Sequential(
            nn.Conv1d(
                in_channels,
                out_channels,
                kernel_size=9,
                stride=4,
                padding=4,
            ),
            nn.BatchNorm1d(out_channels),
            nn.GELU(),
        )

        self.res1 = SameResolutionResidual(
            out_channels,
            kernel_size=5,
            dilation=1,
        )

        self.res3 = SameResolutionResidual(
            out_channels,
            kernel_size=5,
            dilation=3,
        )

    def forward(self, x):
        x = self.down(x)
        x = self.res1(x)
        x = self.res3(x)
        return x


class PerBaseBigVerifierNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv1d(
                5,
                CH_LOCAL,
                kernel_size=11,
                padding=5,
            ),
            nn.BatchNorm1d(CH_LOCAL),
            nn.GELU(),
        )

        self.local1 = SameResolutionResidual(
            CH_LOCAL,
            kernel_size=5,
            dilation=1,
        )

        self.local2 = SameResolutionResidual(
            CH_LOCAL,
            kernel_size=5,
            dilation=2,
        )

        tower = []
        projections = []

        in_ch = CH_LOCAL

        for out_ch in TOWER_CHANNELS:
            tower.append(
                TowerLevel(
                    in_ch,
                    out_ch,
                )
            )

            projections.append(
                nn.Conv1d(
                    out_ch,
                    CH_LOCAL,
                    kernel_size=1,
                )
            )

            in_ch = out_ch

        self.tower = nn.ModuleList(tower)
        self.projections = nn.ModuleList(projections)

        self.fusion_norm = nn.BatchNorm1d(CH_LOCAL)

        self.base_head = nn.Sequential(
            nn.Conv1d(
                CH_LOCAL,
                CH_LOCAL,
                kernel_size=1,
            ),
            nn.GELU(),
            nn.Conv1d(
                CH_LOCAL,
                1,
                kernel_size=1,
            ),
        )

        # Verifier sees:
        # local + tower L3 + L4 + L5 + detached base score
        verifier_in = (
            CH_LOCAL * 4
            + 1
        )

        self.verifier = nn.Sequential(
            nn.Conv1d(
                verifier_in,
                128,
                kernel_size=1,
            ),
            nn.GELU(),
            nn.Conv1d(
                128,
                64,
                kernel_size=1,
            ),
            nn.GELU(),
            nn.Conv1d(
                64,
                1,
                kernel_size=1,
            ),
        )

        # Sparse prior for base log-rate.
        nn.init.constant_(
            self.base_head[-1].bias,
            -8.0,
        )

        # Critical:
        # verifier starts as "no correction".
        nn.init.zeros_(
            self.verifier[-1].weight
        )
        nn.init.zeros_(
            self.verifier[-1].bias
        )

    @staticmethod
    def upsample_to_input(
        x,
        length,
    ):
        # Linear interpolation keeps the multi-scale representation smooth.
        return F.interpolate(
            x,
            size=length,
            mode="linear",
            align_corners=False,
        )

    def forward(self, x):
        length = x.shape[-1]

        stem = self.stem(x)

        local = self.local1(stem)
        local = self.local2(local)

        level_features = []
        projected_full = []

        z = stem

        for level, projection in zip(
            self.tower,
            self.projections,
        ):
            z = level(z)

            level_features.append(z)

            projected = projection(z)

            projected = self.upsample_to_input(
                projected,
                length,
            )

            projected_full.append(
                projected
            )

        # Stable additive multi-scale fusion.
        context_sum = torch.stack(
            projected_full,
            dim=0,
        ).sum(dim=0)

        context_sum = (
            context_sum
            / math.sqrt(
                len(projected_full)
            )
        )

        fused = (
            local
            + context_sum
        )

        fused = F.gelu(
            self.fusion_norm(
                fused
            )
        )

        base_log_rate = self.base_head(
            fused
        )

        # Long-context verifier:
        # explicitly use the three broadest tower levels.
        verifier_features = torch.cat(
            (
                local,
                projected_full[2],
                projected_full[3],
                projected_full[4],
                base_log_rate.detach(),
            ),
            dim=1,
        ).contiguous()

        verifier_logit = self.verifier(
            verifier_features
        )

        correction = (
            CORRECTION_SCALE
            * torch.tanh(
                verifier_logit
                / 3.0
            )
        )

        final_log_rate = (
            base_log_rate
            + correction
        )

        return {
            "final_log_rate": final_log_rate,
            "base_log_rate": base_log_rate,
            "verifier_logit": verifier_logit,
            "correction": correction,
        }


def create_model():
    return PerBaseBigVerifierNet()


model_cpu = create_model()

trainable = sum(
    p.numel()
    for p in model_cpu.parameters()
    if p.requires_grad
)

print(
    "Trainable parameters:",
    f"{trainable:,}",
)

print(
    "Local RF:",
    "~35 bp",
)

print(
    "Tower effective context:",
    "bounded by the 8192-bp input",
)

del model_cpu
gc.collect()


Trainable parameters: 10,391,074
Local RF: ~35 bp
Tower effective context: bounded by the 8192-bp input


60

In [5]:
# ============================================================
# ORIENTATION / CROPPING / LOSSES
# ============================================================

def crop_target(x):
    return x[
        :,
        :,
        CONTEXT_FLANK:
        CONTEXT_FLANK + TARGET_LENGTH
    ].contiguous()


def split_oriented_output(
    output,
    batch_size,
):
    result = {}

    for key, tensor in output.items():
        plus = crop_target(
            tensor[:batch_size]
        )

        minus_reverse = crop_target(
            tensor[batch_size:]
        )

        result[key] = (
            plus,
            minus_reverse,
        )

    return result


def batch_count_targets(batch):
    plus_count = (
        batch.count[
            :,
            0:1,
            :,
        ]
        .float()
        .contiguous()
    )

    minus_count_reverse = (
        torch.flip(
            batch.count[
                :,
                1:2,
                :,
            ],
            dims=[-1],
        )
        .float()
        .contiguous()
    )

    return (
        plus_count,
        minus_count_reverse,
    )


def masked_poisson_nll(
    log_rate,
    count,
    valid,
):
    z = (
        log_rate
        .float()
        .clamp(
            min=-20.0,
            max=15.0,
        )
    )

    y = count.float()
    m = valid.float()

    element = (
        torch.exp(z)
        - y * z
    )

    denom = torch.clamp(
        m.sum(),
        min=1.0,
    )

    return (
        element * m
    ).sum() / denom


def hard_weighted_verifier_bce(
    verifier_logit,
    count,
    valid,
    base_log_rate,
    *,
    hardness_gain=4.0,
    max_pos_weight=64.0,
):
    y = (
        count > 0
    ).float()

    m = valid.float()

    # --------------------------------------------------------
    # 1. Class balancing
    # --------------------------------------------------------

    pos = (
        y * m
    ).sum()

    neg = (
        (1.0 - y)
        * m
    ).sum()

    pos_weight = torch.clamp(
        neg
        / torch.clamp(
            pos,
            min=1.0,
        ),
        min=1.0,
        max=max_pos_weight,
    )

    class_weight = (
        1.0
        + y
        * (
            pos_weight
            - 1.0
        )
    )

    # --------------------------------------------------------
    # 2. Hardness from BASE RATE ERROR
    #
    # Hard negative:
    # count=0, but base predicts high rate.
    #
    # Hard positive:
    # count>0, but base rate is far from observed count.
    # --------------------------------------------------------

    with torch.no_grad():
        base_rate = torch.exp(
            base_log_rate
            .detach()
            .float()
            .clamp(
                min=-20.0,
                max=8.0,
            )
        )

        relative_error = (
            torch.abs(
                base_rate
                - count.float()
            )
            / (
                1.0
                + count.float()
            )
        )

        hardness = torch.clamp(
            relative_error,
            min=0.0,
            max=1.0,
        )

        hard_weight = (
            1.0
            + hardness_gain
            * hardness
        )

    # numerically stable BCEWithLogits:
    # max(x,0) - x*y + log(1 + exp(-abs(x)))
    x = verifier_logit.float()

    bce = (
        torch.clamp(
            x,
            min=0.0,
        )
        - x * y
        + torch.log1p(
            torch.exp(
                -torch.abs(x)
            )
        )
    )

    weight = (
        m
        * class_weight
        * hard_weight
    )

    denom = torch.clamp(
        weight.sum(),
        min=1.0,
    )

    return (
        bce * weight
    ).sum() / denom


In [6]:
# ============================================================
# DIRECTML SMOKE TEST — FORWARD + BACKWARD
# ============================================================

torch.manual_seed(123)

smoke_model = create_model().to(device)

smoke_optimizer = DirectMLAdam(
    smoke_model.parameters(),
    lr=1e-3,
)

smoke_batch = train_generator.sample_batch(
    batch_size=2
)

x_gpu = smoke_batch.x_both_strands.to(device)

(
    plus_count,
    minus_count_reverse,
) = batch_count_targets(
    smoke_batch
)

plus_valid = (
    smoke_batch
    .plus_loss_weight
    .float()
    .to(device)
)

minus_valid = (
    smoke_batch
    .minus_loss_weight_reverse
    .float()
    .to(device)
)

smoke_optimizer.zero_grad(
    set_to_none=True
)

raw_output = smoke_model(
    x_gpu
)

oriented = split_oriented_output(
    raw_output,
    batch_size=2,
)

losses = []

for strand_index, (
    count_cpu,
    valid_gpu,
) in enumerate(
    (
        (
            plus_count,
            plus_valid,
        ),
        (
            minus_count_reverse,
            minus_valid,
        ),
    )
):
    final_rate = oriented[
        "final_log_rate"
    ][strand_index]

    base_rate = oriented[
        "base_log_rate"
    ][strand_index]

    verifier_logit = oriented[
        "verifier_logit"
    ][strand_index]

    count_gpu = count_cpu.to(device)

    final_poisson = masked_poisson_nll(
        final_rate,
        count_gpu,
        valid_gpu,
    )

    base_poisson = masked_poisson_nll(
        base_rate,
        count_gpu,
        valid_gpu,
    )

    verifier_loss = hard_weighted_verifier_bce(
        verifier_logit,
        count_gpu,
        valid_gpu,
        base_rate,
    )

    strand_loss = (
        final_poisson
        + 0.25 * base_poisson
        + 0.05 * verifier_loss
    )

    losses.append(
        strand_loss
    )

smoke_loss = (
    losses[0]
    + losses[1]
) / 2.0

smoke_loss.backward()
smoke_optimizer.step()

print(
    "Smoke loss:",
    float(
        smoke_loss
        .detach()
        .cpu()
        .item()
    ),
)

print(
    "final shape:",
    tuple(
        oriented[
            "final_log_rate"
        ][0].shape
    ),
)

print(
    "correction max abs:",
    float(
        oriented[
            "correction"
        ][0]
        .detach()
        .abs()
        .max()
        .cpu()
        .item()
    ),
)

print("DirectML forward/backward: OK")

del (
    smoke_model,
    smoke_optimizer,
    smoke_batch,
    x_gpu,
    raw_output,
    oriented,
    smoke_loss,
)

gc.collect()


C:\ML_Progs\Runtimes\Miniforge3\envs\epic-eda\Lib\site-packages\torch\nn\functional.py:4073: UserWarning: The operator 'aten::upsample_linear1d.out' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  return torch._C._nn.upsample_linear1d(input, output_size, align_corners, scale_factors)


Smoke loss: 14.695159912109375
final shape: (2, 1, 7168)
correction max abs: 0.0
DirectML forward/backward: OK


42

## Training budget

Мы **не будем делать вывод по 10k**, как сделали с EXP034.

Причина простая: предыдущие эксперименты уже показали, что step budget может менять
качество очень сильно.

Первый meaningful checkpoint здесь — **25k**.

```text
5k   → sanity checkpoint
10k  → trajectory only
25k  → первый нормальный verdict
```

Если модель показывает сильную динамику, следующий запуск — resume до `50k/96k`.


In [7]:
# ============================================================
# TRAIN 25K
# ============================================================

EXPERIMENT = "CNN-EXP-035"
ARCHITECTURE = "perbase_big_verifiernet"

STEPS = 25_000
BATCH_SIZE = 2
LOG_EVERY = 100
SAVE_STEPS = {
    5_000,
    10_000,
    25_000,
}

SEED = 42

BASE_LR = 1e-3
PEAK_LR = 2e-3
WARMUP_FRACTION = 0.05

FINAL_POISSON_WEIGHT = 1.0
BASE_POISSON_WEIGHT = 0.25
VERIFIER_WEIGHT = 0.05

RUN_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / "CNN_EXP035_PERBASE_BIG_VERIFIERNET"
    / "run_001"
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HISTORY_PATH = (
    RUN_DIR
    / "training_history.csv"
)

torch.manual_seed(SEED)

generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=SEED,
)

model = create_model().to(device)

optimizer = DirectMLAdam(
    model.parameters(),
    lr=BASE_LR,
)

model.train()

history = []

ema_total = None
ema_final_poisson = None
ema_verifier = None

started = time.perf_counter()


def learning_rate_for_step(
    step,
):
    warmup_steps = max(
        1,
        int(
            STEPS
            * WARMUP_FRACTION
        ),
    )

    if step <= warmup_steps:
        fraction = (
            step
            / warmup_steps
        )

        return (
            BASE_LR
            + fraction
            * (
                PEAK_LR
                - BASE_LR
            )
        )

    progress = (
        (
            step
            - warmup_steps
        )
        / max(
            1,
            STEPS
            - warmup_steps,
        )
    )

    cosine = (
        0.5
        * (
            1.0
            + math.cos(
                math.pi
                * progress
            )
        )
    )

    return (
        BASE_LR
        * 0.10
        + (
            PEAK_LR
            - BASE_LR
            * 0.10
        )
        * cosine
    )


def set_lr(
    optimizer,
    lr,
):
    for group in optimizer.param_groups:
        group["lr"] = lr


def model_state_on_cpu(model):
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor
        in model.state_dict().items()
    }


def tree_to_cpu(value):
    if torch.is_tensor(value):
        return (
            value.detach()
            .cpu()
            .clone()
        )

    if isinstance(value, dict):
        return {
            key: tree_to_cpu(item)
            for key, item
            in value.items()
        }

    if isinstance(value, list):
        return [
            tree_to_cpu(item)
            for item in value
        ]

    if isinstance(value, tuple):
        return tuple(
            tree_to_cpu(item)
            for item in value
        )

    return value


def checkpoint_path(step):
    return (
        RUN_DIR
        / f"checkpoint_step_{step:06d}.pt"
    )


def save_checkpoint(step):
    checkpoint = {
        "experiment": EXPERIMENT,
        "architecture": ARCHITECTURE,
        "step": int(step),
        "model_state_dict": model_state_on_cpu(
            model
        ),
        "optimizer_state_dict": tree_to_cpu(
            optimizer.state_dict()
        ),
        "generator_rng_state": copy.deepcopy(
            generator
            .rng
            .bit_generator
            .state
        ),
        "torch_rng_state": torch.get_rng_state(),
        "profile_config": {
            "target_length": TARGET_LENGTH,
            "context_flank": CONTEXT_FLANK,
            "input_length": INPUT_LENGTH,
            "positive_branch_fraction": 0.5,
        },
        "model_config": {
            "ch_local": CH_LOCAL,
            "tower_channels": list(
                TOWER_CHANNELS
            ),
            "correction_scale": CORRECTION_SCALE,
        },
        "training_config": {
            "batch_size": BATCH_SIZE,
            "steps": STEPS,
            "base_lr": BASE_LR,
            "peak_lr": PEAK_LR,
            "warmup_fraction": WARMUP_FRACTION,
            "final_poisson_weight": FINAL_POISSON_WEIGHT,
            "base_poisson_weight": BASE_POISSON_WEIGHT,
            "verifier_weight": VERIFIER_WEIGHT,
            "seed": SEED,
        },
        "ema_total": float(ema_total),
        "ema_final_poisson": float(
            ema_final_poisson
        ),
        "ema_verifier": float(
            ema_verifier
        ),
        "elapsed_minutes": float(
            (
                time.perf_counter()
                - started
            )
            / 60.0
        ),
    }

    path = checkpoint_path(step)

    torch.save(
        checkpoint,
        path,
    )

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_PATH,
        index=False,
    )

    print()
    print(
        f"=== CHECKPOINT {step:,} SAVED ==="
    )
    print(
        "EMA total:",
        round(
            float(ema_total),
            6,
        ),
    )
    print(
        "EMA verifier:",
        round(
            float(ema_verifier),
            6,
        ),
    )
    print(
        "Path:",
        path,
    )
    print()


for step in range(
    1,
    STEPS + 1,
):
    lr = learning_rate_for_step(
        step
    )

    set_lr(
        optimizer,
        lr,
    )

    batch = generator.sample_batch(
        batch_size=BATCH_SIZE
    )

    x_gpu = (
        batch
        .x_both_strands
        .to(device)
    )

    (
        plus_count,
        minus_count_reverse,
    ) = batch_count_targets(
        batch
    )

    plus_valid = (
        batch
        .plus_loss_weight
        .float()
        .to(device)
    )

    minus_valid = (
        batch
        .minus_loss_weight_reverse
        .float()
        .to(device)
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    raw_output = model(
        x_gpu
    )

    oriented = split_oriented_output(
        raw_output,
        batch_size=BATCH_SIZE,
    )

    strand_total_losses = []
    strand_final_losses = []
    strand_base_losses = []
    strand_verifier_losses = []

    for strand_index, (
        count_cpu,
        valid_gpu,
    ) in enumerate(
        (
            (
                plus_count,
                plus_valid,
            ),
            (
                minus_count_reverse,
                minus_valid,
            ),
        )
    ):
        count_gpu = count_cpu.to(
            device
        )

        final_rate = oriented[
            "final_log_rate"
        ][strand_index]

        base_rate = oriented[
            "base_log_rate"
        ][strand_index]

        verifier_logit = oriented[
            "verifier_logit"
        ][strand_index]

        final_poisson = masked_poisson_nll(
            final_rate,
            count_gpu,
            valid_gpu,
        )

        base_poisson = masked_poisson_nll(
            base_rate,
            count_gpu,
            valid_gpu,
        )

        verifier_loss = hard_weighted_verifier_bce(
            verifier_logit,
            count_gpu,
            valid_gpu,
            base_rate,
        )

        strand_total = (
            FINAL_POISSON_WEIGHT
            * final_poisson
            + BASE_POISSON_WEIGHT
            * base_poisson
            + VERIFIER_WEIGHT
            * verifier_loss
        )

        strand_total_losses.append(
            strand_total
        )

        strand_final_losses.append(
            final_poisson
        )

        strand_base_losses.append(
            base_poisson
        )

        strand_verifier_losses.append(
            verifier_loss
        )

    total_loss = (
        strand_total_losses[0]
        + strand_total_losses[1]
    ) / 2.0

    final_poisson_loss = (
        strand_final_losses[0]
        + strand_final_losses[1]
    ) / 2.0

    base_poisson_loss = (
        strand_base_losses[0]
        + strand_base_losses[1]
    ) / 2.0

    verifier_loss = (
        strand_verifier_losses[0]
        + strand_verifier_losses[1]
    ) / 2.0

    total_value = float(
        total_loss
        .detach()
        .cpu()
        .item()
    )

    if not math.isfinite(
        total_value
    ):
        raise FloatingPointError(
            f"Non-finite loss at step {step}"
        )

    total_loss.backward()
    optimizer.step()

    final_value = float(
        final_poisson_loss
        .detach()
        .cpu()
        .item()
    )

    base_value = float(
        base_poisson_loss
        .detach()
        .cpu()
        .item()
    )

    verifier_value = float(
        verifier_loss
        .detach()
        .cpu()
        .item()
    )

    if ema_total is None:
        ema_total = total_value
        ema_final_poisson = final_value
        ema_verifier = verifier_value
    else:
        decay = 0.95

        ema_total = (
            decay * ema_total
            + (1.0 - decay)
            * total_value
        )

        ema_final_poisson = (
            decay * ema_final_poisson
            + (1.0 - decay)
            * final_value
        )

        ema_verifier = (
            decay * ema_verifier
            + (1.0 - decay)
            * verifier_value
        )

    if (
        step == 1
        or step % LOG_EVERY == 0
    ):
        correction_abs = float(
            oriented[
                "correction"
            ][0]
            .detach()
            .abs()
            .mean()
            .cpu()
            .item()
        )

        elapsed = (
            time.perf_counter()
            - started
        )

        history.append({
            "step": int(step),
            "lr": float(lr),
            "loss": total_value,
            "final_poisson": final_value,
            "base_poisson": base_value,
            "verifier_bce": verifier_value,
            "ema_total": float(
                ema_total
            ),
            "ema_final_poisson": float(
                ema_final_poisson
            ),
            "ema_verifier": float(
                ema_verifier
            ),
            "mean_abs_correction": (
                correction_abs
            ),
            "elapsed_minutes": float(
                elapsed
                / 60.0
            ),
        })

        print(
            f"step={step:6d} "
            f"lr={lr:.6g} "
            f"loss={total_value:.5f} "
            f"finalP={final_value:.5f} "
            f"baseP={base_value:.5f} "
            f"V={verifier_value:.5f} "
            f"|corr|={correction_abs:.4f}"
        )

    if step in SAVE_STEPS:
        save_checkpoint(
            step
        )

pd.DataFrame(
    history
).to_csv(
    HISTORY_PATH,
    index=False,
)

print()
print(
    "Training finished in",
    round(
        (
            time.perf_counter()
            - started
        )
        / 60.0,
        2,
    ),
    "min",
)

print(
    "Final checkpoint:",
    checkpoint_path(
        STEPS
    ),
)


step=     1 lr=0.0010008 loss=36.16070 finalP=28.90835 baseP=28.90835 V=0.50516 |corr|=0.0000
step=   100 lr=0.00108 loss=8.36851 finalP=6.32653 baseP=7.81237 V=1.77772 |corr|=1.5944
step=   200 lr=0.00116 loss=1.25279 finalP=-0.13053 baseP=4.55786 V=4.87702 |corr|=1.7467
step=   300 lr=0.00124 loss=-4.85864 finalP=-5.11901 baseP=0.60104 V=2.20219 |corr|=1.9553
step=   400 lr=0.00132 loss=1.20603 finalP=0.42027 baseP=0.30986 V=14.16590 |corr|=1.9899
step=   500 lr=0.0014 loss=0.82322 finalP=0.28451 baseP=1.08572 V=5.34562 |corr|=1.9970
step=   600 lr=0.00148 loss=7.01980 finalP=3.82948 baseP=3.91546 V=44.22913 |corr|=2.0000
step=   700 lr=0.00156 loss=0.63124 finalP=-2.23895 baseP=10.96216 V=2.59307 |corr|=1.9901
step=   800 lr=0.00164 loss=-1.03916 finalP=-2.00481 baseP=1.89733 V=9.82640 |corr|=1.9873
step=   900 lr=0.00172 loss=9.74555 finalP=8.59203 baseP=2.54146 V=10.36303 |corr|=1.9162
step=  1000 lr=0.0018 loss=1.97189 finalP=0.82193 baseP=0.50678 V=20.46540 |corr|=1.5491
step=  

## Full holdout — memory-safe

В EXP034 kernel падал не из-за модели, а из-за тяжёлого evaluation pipeline.

Поэтому здесь validation сразу сделана в безопасном режиме:

- `batch_size=1`;
- GPU output немедленно переносится на CPU;
- никаких огромных arrays со всеми negatives;
- AP считается streaming histogram;
- в RAM полностью хранятся только positive scores для Spearman;
- одновременно считаются **BASE** и **FINAL**, чтобы увидеть чистый эффект verifier.

Главный вопрос EXP035:

```text
FINAL AP > BASE AP ?
FINAL Spearman >= BASE Spearman ?
```

Если да — Context Verifier действительно исправляет часть ошибок backbone.


In [8]:
# ============================================================
# LOAD 25K CHECKPOINT FOR SAFE EVALUATION
# ============================================================

EVAL_STEP = 25_000

CHECKPOINT_PATH = checkpoint_path(
    EVAL_STEP
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

validation_model = create_model()

validation_model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

validation_model = validation_model.to(
    device
)

validation_model.eval()

del checkpoint
gc.collect()

print(
    "Loaded:",
    CHECKPOINT_PATH,
)

print(
    "Validation tiles:",
    len(validation_tiles),
)


Loaded: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP035_PERBASE_BIG_VERIFIERNET\run_001\checkpoint_step_025000.pt
Validation tiles: 7440


In [9]:
# ============================================================
# SAFE STREAMING METRIC HELPERS
# ============================================================

RAW_LOG_MIN = -20.0
RAW_LOG_MAX = 15.0
RAW_AP_BINS = 1_000_001


def raw_lograte_bin(scores):
    clipped = np.clip(
        scores,
        RAW_LOG_MIN,
        RAW_LOG_MAX,
    )

    scaled = (
        (
            clipped
            - RAW_LOG_MIN
        )
        / (
            RAW_LOG_MAX
            - RAW_LOG_MIN
        )
    )

    return np.rint(
        scaled
        * (
            RAW_AP_BINS - 1
        )
    ).astype(np.int32)


def update_histogram(
    hist,
    indices,
):
    values, counts = np.unique(
        indices,
        return_counts=True,
    )

    hist[
        values
    ] += counts.astype(
        np.int64
    )


def ap_from_histograms(
    positive_hist,
    total_hist,
):
    positives = int(
        positive_hist.sum()
    )

    if positives <= 0:
        return float("nan")

    pos_desc = (
        positive_hist[
            ::-1
        ]
    )

    total_desc = (
        total_hist[
            ::-1
        ]
    )

    tp = np.cumsum(
        pos_desc,
        dtype=np.int64,
    )

    predicted_positive = np.cumsum(
        total_desc,
        dtype=np.int64,
    )

    precision = (
        tp
        / np.maximum(
            predicted_positive,
            1,
        )
    )

    return float(
        np.sum(
            (
                pos_desc
                / positives
            )
            * precision
        )
    )


In [10]:
# ============================================================
# FULL HOLDOUT — BASE vs FINAL
# ============================================================

EVAL_BATCH_SIZE = 1
PROGRESS_EVERY = 100

base_total_hist = np.zeros(
    RAW_AP_BINS,
    dtype=np.int64,
)

base_positive_hist = np.zeros(
    RAW_AP_BINS,
    dtype=np.int64,
)

final_total_hist = np.zeros(
    RAW_AP_BINS,
    dtype=np.int64,
)

final_positive_hist = np.zeros(
    RAW_AP_BINS,
    dtype=np.int64,
)

verifier_total_hist = np.zeros(
    RAW_AP_BINS,
    dtype=np.int64,
)

verifier_positive_hist = np.zeros(
    RAW_AP_BINS,
    dtype=np.int64,
)

positive_base_chunks = []
positive_final_chunks = []
positive_verifier_chunks = []
positive_count_chunks = []

correction_positive_sum = 0.0
correction_negative_sum = 0.0
correction_positive_n = 0
correction_negative_n = 0

positions = 0
started_eval = time.perf_counter()

with torch.no_grad():

    for batch_no, start in enumerate(
        range(
            0,
            len(validation_tiles),
            EVAL_BATCH_SIZE,
        ),
        1,
    ):
        selected = validation_tiles.iloc[
            start:
            start + EVAL_BATCH_SIZE
        ]

        batch = validation_generator.make_batch(
            selected
        )

        B = len(selected)

        x_gpu = (
            batch
            .x_both_strands
            .to(device)
        )

        raw_output = validation_model(
            x_gpu
        )

        oriented = split_oriented_output(
            raw_output,
            batch_size=B,
        )

        # Move everything to CPU FIRST.
        plus_base = (
            oriented[
                "base_log_rate"
            ][0][
                :,
                0,
                :,
            ]
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32,
                copy=True,
            )
        )

        minus_base_reverse = (
            oriented[
                "base_log_rate"
            ][1][
                :,
                0,
                :,
            ]
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32,
                copy=True,
            )
        )

        plus_final = (
            oriented[
                "final_log_rate"
            ][0][
                :,
                0,
                :,
            ]
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32,
                copy=True,
            )
        )

        minus_final_reverse = (
            oriented[
                "final_log_rate"
            ][1][
                :,
                0,
                :,
            ]
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32,
                copy=True,
            )
        )

        plus_verifier = (
            oriented[
                "verifier_logit"
            ][0][
                :,
                0,
                :,
            ]
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32,
                copy=True,
            )
        )

        minus_verifier_reverse = (
            oriented[
                "verifier_logit"
            ][1][
                :,
                0,
                :,
            ]
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32,
                copy=True,
            )
        )

        del (
            x_gpu,
            raw_output,
            oriented,
        )

        minus_base = (
            minus_base_reverse[
                :,
                ::-1
            ]
            .copy()
        )

        minus_final = (
            minus_final_reverse[
                :,
                ::-1
            ]
            .copy()
        )

        minus_verifier = (
            minus_verifier_reverse[
                :,
                ::-1
            ]
            .copy()
        )

        base_score = np.stack(
            (
                plus_base,
                minus_base,
            ),
            axis=1,
        )

        final_score = np.stack(
            (
                plus_final,
                minus_final,
            ),
            axis=1,
        )

        verifier_score = np.stack(
            (
                plus_verifier,
                minus_verifier,
            ),
            axis=1,
        )

        count = (
            batch
            .count
            .numpy()
            .astype(
                np.float32,
                copy=False,
            )
        )

        mask = (
            batch
            .mask
            .numpy()
            .astype(
                bool,
                copy=False,
            )
        )

        valid_base = base_score[
            mask
        ]

        valid_final = final_score[
            mask
        ]

        valid_verifier = verifier_score[
            mask
        ]

        valid_count = count[
            mask
        ]

        positive = (
            valid_count > 0
        )

        base_idx = raw_lograte_bin(
            valid_base
        )

        final_idx = raw_lograte_bin(
            valid_final
        )

        verifier_idx = raw_lograte_bin(
            valid_verifier
        )

        update_histogram(
            base_total_hist,
            base_idx,
        )

        update_histogram(
            final_total_hist,
            final_idx,
        )

        update_histogram(
            verifier_total_hist,
            verifier_idx,
        )

        if positive.any():
            update_histogram(
                base_positive_hist,
                base_idx[
                    positive
                ],
            )

            update_histogram(
                final_positive_hist,
                final_idx[
                    positive
                ],
            )

            update_histogram(
                verifier_positive_hist,
                verifier_idx[
                    positive
                ],
            )

            positive_base_chunks.append(
                valid_base[
                    positive
                ].astype(
                    np.float32,
                    copy=True,
                )
            )

            positive_final_chunks.append(
                valid_final[
                    positive
                ].astype(
                    np.float32,
                    copy=True,
                )
            )

            positive_verifier_chunks.append(
                valid_verifier[
                    positive
                ].astype(
                    np.float32,
                    copy=True,
                )
            )

            positive_count_chunks.append(
                valid_count[
                    positive
                ].astype(
                    np.float32,
                    copy=True,
                )
            )

        correction = (
            valid_final
            - valid_base
        )

        correction_positive_sum += float(
            correction[
                positive
            ].sum(
                dtype=np.float64
            )
        )

        correction_negative_sum += float(
            correction[
                ~positive
            ].sum(
                dtype=np.float64
            )
        )

        correction_positive_n += int(
            positive.sum()
        )

        correction_negative_n += int(
            (
                ~positive
            ).sum()
        )

        positions += int(
            mask.sum()
        )

        del (
            batch,
            selected,
            plus_base,
            minus_base_reverse,
            minus_base,
            plus_final,
            minus_final_reverse,
            minus_final,
            plus_verifier,
            minus_verifier_reverse,
            minus_verifier,
            base_score,
            final_score,
            verifier_score,
            count,
            mask,
            valid_base,
            valid_final,
            valid_verifier,
            valid_count,
            positive,
            base_idx,
            final_idx,
            verifier_idx,
            correction,
        )

        if batch_no % 25 == 0:
            gc.collect()

        finished = (
            start + B
            >= len(validation_tiles)
        )

        if (
            batch_no == 1
            or batch_no % PROGRESS_EVERY == 0
            or finished
        ):
            elapsed = (
                time.perf_counter()
                - started_eval
            )

            print(
                f"tiles={start + B:>6}"
                f"/{len(validation_tiles)} "
                f"positions={positions:>12,} "
                f"positions/s="
                f"{positions / elapsed:,.0f}"
            )

print()
print("FULL HOLDOUT INFERENCE FINISHED")


tiles=     1/7440 positions=       8,582 positions/s=81,786
tiles=   100/7440 positions=     922,930 positions/s=204,081
tiles=   200/7440 positions=   1,910,796 positions/s=213,690
tiles=   300/7440 positions=   3,072,212 positions/s=225,720
tiles=   400/7440 positions=   4,152,314 positions/s=229,083
tiles=   500/7440 positions=   5,367,588 positions/s=234,076
tiles=   600/7440 positions=   6,568,488 positions/s=237,683
tiles=   700/7440 positions=   7,718,612 positions/s=239,030
tiles=   800/7440 positions=   8,946,934 positions/s=241,185
tiles=   900/7440 positions=  10,121,628 positions/s=242,318
tiles=  1000/7440 positions=  11,326,822 positions/s=243,073
tiles=  1100/7440 positions=  12,520,928 positions/s=243,732
tiles=  1200/7440 positions=  13,733,454 positions/s=244,066
tiles=  1300/7440 positions=  14,952,956 positions/s=245,148
tiles=  1400/7440 positions=  16,095,138 positions/s=245,253
tiles=  1500/7440 positions=  17,275,824 positions/s=245,516
tiles=  1600/7440 positio

In [11]:
# ============================================================
# METRICS
# ============================================================

base_ap = ap_from_histograms(
    base_positive_hist,
    base_total_hist,
)

final_ap = ap_from_histograms(
    final_positive_hist,
    final_total_hist,
)

verifier_ap = ap_from_histograms(
    verifier_positive_hist,
    verifier_total_hist,
)

positive_base = np.concatenate(
    positive_base_chunks
)

positive_final = np.concatenate(
    positive_final_chunks
)

positive_verifier = np.concatenate(
    positive_verifier_chunks
)

positive_count = np.concatenate(
    positive_count_chunks
)

target_rank = rankdata(
    positive_count,
    method="dense",
)

base_spearman = float(
    pearsonr(
        rankdata(
            positive_base,
            method="dense",
        ),
        target_rank,
    )[0]
)

final_spearman = float(
    pearsonr(
        rankdata(
            positive_final,
            method="dense",
        ),
        target_rank,
    )[0]
)

verifier_spearman = float(
    pearsonr(
        rankdata(
            positive_verifier,
            method="dense",
        ),
        target_rank,
    )[0]
)

mean_corr_positive = (
    correction_positive_sum
    / max(
        correction_positive_n,
        1,
    )
)

mean_corr_negative = (
    correction_negative_sum
    / max(
        correction_negative_n,
        1,
    )
)

metrics = pd.DataFrame([
    {
        "model_output": "BASE",
        "per_base_AP": base_ap,
        "EPIC_Spearman": base_spearman,
    },
    {
        "model_output": "VERIFIER_ONLY",
        "per_base_AP": verifier_ap,
        "EPIC_Spearman": verifier_spearman,
    },
    {
        "model_output": "FINAL_BASE_PLUS_VERIFIER",
        "per_base_AP": final_ap,
        "EPIC_Spearman": final_spearman,
    },
])

display(metrics)

print()
print(
    "Δ AP FINAL - BASE:",
    final_ap - base_ap,
)

print(
    "Δ Spearman FINAL - BASE:",
    final_spearman - base_spearman,
)

print()
print(
    "Mean correction on positive bp:",
    mean_corr_positive,
)

print(
    "Mean correction on negative bp:",
    mean_corr_negative,
)


,model_output,per_base_AP,EPIC_Spearman
0,BASE,0.001268,0.114229
1,VERIFIER_ONLY,0.000670,-0.047689
2,FINAL_BASE_PLUS_VERIFIER,0.001267,0.113784



Δ AP FINAL - BASE: -1.3565094344589104e-06
Δ Spearman FINAL - BASE: -0.00044480711221694946

Mean correction on positive bp: 0.13614164202167262
Mean correction on negative bp: 0.13712815391715974


In [12]:
# ============================================================
# SAVE ONE JSON
# ============================================================

RESULTS_JSON = (
    RUN_DIR
    / "EXP035_RESULTS_25K.json"
)

payload = {
    "experiment": {
        "name": EXPERIMENT,
        "architecture": ARCHITECTURE,
        "checkpoint_step": EVAL_STEP,
        "prediction_unit": (
            "single nucleotide x strand"
        ),
        "regional_target_used": False,
        "input_length": INPUT_LENGTH,
        "target_length": TARGET_LENGTH,
        "context_flank": CONTEXT_FLANK,
        "input_channels": 5,
        "ch_local": CH_LOCAL,
        "tower_channels": list(
            TOWER_CHANNELS
        ),
        "correction_scale": (
            CORRECTION_SCALE
        ),
    },

    "metrics": (
        metrics
        .where(
            pd.notnull(metrics),
            None,
        )
        .to_dict(
            orient="records"
        )
    ),

    "verifier_effect": {
        "delta_AP_final_minus_base": float(
            final_ap
            - base_ap
        ),
        "delta_Spearman_final_minus_base": float(
            final_spearman
            - base_spearman
        ),
        "mean_correction_positive_bp": float(
            mean_corr_positive
        ),
        "mean_correction_negative_bp": float(
            mean_corr_negative
        ),
    },

    "reference": {
        "EXP034_10k_per_base_AP_raw": (
            0.0014684401907595272
        ),
        "EXP034_10k_EPIC_Spearman_raw": (
            0.10031352937221527
        ),
        "EPIC_solution_big_AP": (
            0.131650
        ),
        "EPIC_solution_big_Spearman": (
            0.391000
        ),
    },

    "interpretation_contract": [
        (
            "FINAL > BASE tests whether the context verifier "
            "adds useful residual information."
        ),
        (
            "This experiment never trains on 32-bp regional labels."
        ),
        (
            "Do not judge the architecture from 5k/10k alone; "
            "25k is the first planned verdict."
        ),
        (
            "If promising, resume the same checkpoint toward 50k/96k."
        ),
    ],
}

RESULTS_JSON.write_text(
    json.dumps(
        payload,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    ),
    encoding="utf-8",
)

print()
print("=" * 72)
print("ГОТОВО")
print("=" * 72)
print()
print("Пришли мне этот файл:")
print()
print(RESULTS_JSON)



ГОТОВО

Пришли мне этот файл:

D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP035_PERBASE_BIG_VERIFIERNET\run_001\EXP035_RESULTS_25K.json
